# Fine-tune Thai transformers for YouTube virality prediction

Run this notebook on **Google Colab** (T4 free / A100 Pro+) or **Kaggle** (T4 ×2 / P100).

Workflow:
1. Mount Drive (or upload directly)
2. Clone the repo
3. Install deps via `uv`
4. Upload the processed parquet (`dataset_with_labels.parquet`) generated locally by `scripts/prepare_data.py`
5. Fine-tune the 3 encoders one at a time
6. Download predictions back to local for `scripts/evaluate.py`

Estimated wall time on Colab T4 (16 GB):
- WangchanBERTa (105 M, full FT, 5 epochs) → **~25 min**
- PhayaThaiBERT (278 M, full FT, 4 epochs) → **~1 h 10 min**
- XLM-RoBERTa-large (560 M, full FT, 4 epochs) → **~2 h 30 min**

On Colab Pro+ A100 these are roughly 3 – 4 × faster.

In [ ]:
# 0) Verify GPU
!nvidia-smi

In [ ]:
# 1) Clone the repo
!git clone https://github.com/MNupakorn/thesis-youtube-virality-thai.git
%cd thesis-youtube-virality-thai
!git pull

In [ ]:
# 2) Install deps
!pip install -q -U uv
!uv pip install --system -q -e ".[dev]"

In [ ]:
# 3) Upload the locally-prepared dataset
#    Run on your local machine first:
#       python scripts/prepare_data.py --skip-sentiment --skip-embeddings
#    Then drag `data/processed/dataset_with_labels.parquet` into the upload prompt below.
import os, shutil
os.makedirs('data/processed', exist_ok=True)
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.parquet'):
        shutil.move(fname, f'data/processed/{fname}')
        print(f'moved -> data/processed/{fname}')

In [ ]:
# 4) (Optional) HuggingFace login — only needed if you hit rate limits
# from huggingface_hub import login
# login()

In [ ]:
# 5a) WangchanBERTa  (~25 min on T4)
!python scripts/train_transformer.py --model wangchanberta --config configs/train.yaml

In [ ]:
# 5b) PhayaThaiBERT  (~1 h on T4, ~20 min on A100)
!python scripts/train_transformer.py --model phayathaibert --config configs/train.yaml

In [ ]:
# 5c) XLM-RoBERTa-large  (~2.5 h on T4, ~45 min on A100)
!python scripts/train_transformer.py --model xlm-roberta-large --config configs/train.yaml

In [ ]:
# 6) (Optional) Compute title transformer embeddings on GPU + train hybrid model
!python scripts/train_hybrid.py --config configs/train.yaml

In [ ]:
# 7) Run the full evaluation incl. McNemar's + Cochran's Q
!python scripts/evaluate.py --config configs/eval.yaml

In [ ]:
# 8) Tar up + download artifacts back to local
!tar czf artifacts.tgz reports/ mlruns/
from google.colab import files
files.download('artifacts.tgz')